# Explore S3: `batch3_data_10k/10k_data`
Lists every subfolder and file, then shows the schema, row count and sample rows of each table.

Credentials come from `.env` (gitignored). They're temporary STS credentials, so paste fresh ones into `.env` when they expire.

In [7]:
import os
import pandas as pd
import pyarrow.parquet as pq
from pyarrow import fs
from dotenv import load_dotenv

load_dotenv(override=True)  # reads .env next to this notebook

BUCKET = "ld-paypalb1-data-448049797912"
PREFIX = "Learning and Development/Batches/batch3_data_10k/10k_data"
ROOT = f"{BUCKET}/{PREFIX}"

s3 = fs.S3FileSystem(
    access_key=os.environ["AWS_ACCESS_KEY_ID"],
    secret_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    session_token=os.environ.get("AWS_SESSION_TOKEN"),
    region=os.getenv("AWS_DEFAULT_REGION", "us-east-1"),
)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. Subfolders and files

In [8]:
infos = [i for i in s3.get_file_info(fs.FileSelector(ROOT, recursive=True)) if i.type == fs.FileType.File]

files = pd.DataFrame({
    "path": [i.path for i in infos],
    "size_mb": [round(i.size / 1024**2, 2) for i in infos],
})
rel = files["path"].str[len(ROOT) + 1:]
files["subfolder"] = rel.str.split("/").str[0]
files["file"] = rel.str.split("/").str[-1]

print(f"{len(files)} files, {files.size_mb.sum():.1f} MB total\n")
files.groupby("subfolder").agg(n_files=("file", "count"), total_mb=("size_mb", "sum")).round(2)

56 files, 569.3 MB total



,n_files,total_mb
subfolder,,
collections_case,5,0.05
collections_case_snapshot,5,0.10
customer_profile,5,0.25
customer_snapshot,5,0.75
fraud_case,5,0.05
fraud_case_snapshot,5,0.05
interaction_event,5,161.79
merchant,1,0.04
product_event,5,1.85


In [9]:
for sub, grp in files.groupby("subfolder"):
    print(f"📁 {sub}/")
    for _, r in grp.iterrows():
        print(f"    {r.file:<25} {r.size_mb:>8.2f} MB")

📁 collections_case/
    c0120000.parquet              0.01 MB
    c0122000.parquet              0.01 MB
    c0124000.parquet              0.01 MB
    c0126000.parquet              0.01 MB
    c0128000.parquet              0.01 MB
📁 collections_case_snapshot/
    c0120000.parquet              0.02 MB
    c0122000.parquet              0.02 MB
    c0124000.parquet              0.02 MB
    c0126000.parquet              0.02 MB
    c0128000.parquet              0.02 MB
📁 customer_profile/
    c0120000.parquet              0.05 MB
    c0122000.parquet              0.05 MB
    c0124000.parquet              0.05 MB
    c0126000.parquet              0.05 MB
    c0128000.parquet              0.05 MB
📁 customer_snapshot/
    c0120000.parquet              0.15 MB
    c0122000.parquet              0.15 MB
    c0124000.parquet              0.15 MB
    c0126000.parquet              0.15 MB
    c0128000.parquet              0.15 MB
📁 fraud_case/
    c0120000.parquet              0.01 MB
    c0122000.p

## 2. Data in each subfolder
For each subfolder: the total row count across all files and the schema, both read from parquet metadata so nothing large is downloaded. Then the first 5 rows of the first file, read from one row group only.

In [10]:
N_SAMPLE = 5

for sub, grp in files.groupby("subfolder"):
    paths = list(grp["path"])
    n_rows = sum(pq.read_metadata(p, filesystem=s3).num_rows for p in paths)
    pf = pq.ParquetFile(paths[0], filesystem=s3)

    print("=" * 100)
    print(f"📁 {sub}  |  {len(paths)} file(s)  |  {n_rows:,} rows  |  {pf.schema_arrow.__len__()} columns")
    print("=" * 100)
    print(pf.schema_arrow.to_string(show_schema_metadata=False))
    sample = pf.read_row_group(0).slice(0, N_SAMPLE).to_pandas()
    display(sample)

📁 collections_case  |  5 file(s)  |  610 rows  |  15 columns
collection_case_id: string not null
product_id: string not null
customer_id: string not null
case_open_timestamp: timestamp[us] not null
case_open_dpd: int32 not null
outstanding_at_assignment: decimal128(18, 2) not null
strategy_segment_code: string not null
promise_made_timestamp: timestamp[us]
promise_amount: decimal128(18, 2)
promise_date: date32[day]
arrangement_type: string
arrangement_start_date: date32[day]
number_of_instalments: int32
instalment_amount: decimal128(18, 2)
total_arrangement_amount: decimal128(18, 2)


,collection_case_id,product_id,customer_id,case_open_timestamp,case_open_dpd,outstanding_at_assignment,strategy_segment_code,promise_made_timestamp,promise_amount,promise_date,arrangement_type,arrangement_start_date,number_of_instalments,instalment_amount,total_arrangement_amount
0,COL0720017000,PRD0160017002,CIN0009920017,2028-09-02 08:51:04.666916,34,1690.39,EARLY_STAGE_SELF_CURE,2028-09-02,1463.06,2028-09-12,LONG_TERM_RESTRUCTURE,2028-09-03,7.0,241.48,1690.36
1,COL0720019000,PRD0160019002,CIN0009920019,2028-05-19 07:55:21.863356,38,1579.60,HIGH_BALANCE_PRIORITY,NaT,None,None,None,None,NaN,None,None
2,COL0720019001,PRD0160019002,CIN0009920019,2028-08-24 07:50:34.881335,32,2693.96,HIGH_BALANCE_PRIORITY,NaT,None,None,None,None,NaN,None,None
3,COL0720023000,PRD0160023002,CIN0009920023,2026-05-15 08:52:03.576747,34,1175.52,REPEAT_DELINQUENT,2026-05-29,974.79,2026-06-08,LONG_TERM_RESTRUCTURE,2026-05-23,7.0,167.93,1175.51
4,COL0720042000,PRD0160042002,CIN0009920042,2026-06-24 09:13:18.745536,31,725.22,EARLY_STAGE_SELF_CURE,NaT,None,None,None,None,NaN,None,None


📁 collections_case_snapshot  |  5 file(s)  |  4,394 rows  |  19 columns
snapshot_date: date32[day] not null
collection_case_id: string not null
collection_stage: string not null
case_status: string not null
case_close_reason: string
amount_collected_to_date: decimal128(18, 2) not null
promise_status: string
arrangement_status: string
dpd_at_action: int32 not null
action_id: string
action_timestamp: timestamp[us]
action_channel: string
contact_outcome: string
collections_action_type: string
collections_outcome: string
action_amount_collected: decimal128(18, 2)
writeoff_amount: decimal128(18, 2)
recovery_amount: decimal128(18, 2)
recovery_source: string


,snapshot_date,collection_case_id,collection_stage,case_status,case_close_reason,amount_collected_to_date,promise_status,arrangement_status,dpd_at_action,action_id,action_timestamp,action_channel,contact_outcome,collections_action_type,collections_outcome,action_amount_collected,writeoff_amount,recovery_amount,recovery_source
0,2028-09-02,COL0720017000,SOFT,OPEN,None,0.00,None,None,34,COLA012701700000,2028-09-02 20:23:01.752102,OUTBOUND_CALL,PROMISE_OBTAINED,NEGOTIATION,PROMISE,None,None,None,None
1,2028-09-09,COL0720017000,SOFT,OPEN,None,0.00,None,ACTIVE,41,None,NaT,None,None,None,None,None,None,None,None
2,2028-09-16,COL0720017000,SOFT,OPEN,None,1463.06,KEPT,ACTIVE,0,COLA012701700001,2028-09-11 15:13:17.224239,OUTBOUND_CALL,NO_CONTACT,NEGOTIATION,UNRESOLVED,None,None,None,None
3,2028-05-19,COL0720019000,SOFT,OPEN,None,0.00,None,None,38,COLA012701900000,2028-05-19 12:42:58.719313,EMAIL_NOTICE,DISPUTE_RAISED,REMINDER,DISPUTE,None,None,None,None
4,2028-05-26,COL0720019000,SOFT,OPEN,None,0.00,None,None,45,COLA012701900001,2028-05-20 13:47:56.510251,EMAIL_NOTICE,NO_CONTACT,REMINDER,UNRESOLVED,None,None,None,None


📁 customer_profile  |  5 file(s)  |  10,000 rows  |  19 columns
customer_id: string not null
cust_segment: string not null
relationship_open_date: date32[day] not null
acc_cre_ts: timestamp[us] not null
relationship_close_date: date32[day]
customer_is_active: bool not null
is_cust_stat_clr_y_n: string not null
home_currency: string not null
nationality_country_code: string
residence_country: string not null
residence_state: string
employment_status: string
occupation_category: string
income_band: string
kyc_status: string not null
kyc_risk_rating: string
kyc_last_review_date: date32[day]
preferred_contact_channel: string
marketing_consent: bool not null


,customer_id,cust_segment,relationship_open_date,acc_cre_ts,relationship_close_date,customer_is_active,is_cust_stat_clr_y_n,home_currency,nationality_country_code,residence_country,residence_state,employment_status,occupation_category,income_band,kyc_status,kyc_risk_rating,kyc_last_review_date,preferred_contact_channel,marketing_consent
0,CIN0009920000,AFFLUENT,2016-02-28,2016-02-28 12:23:00.690962,None,True,Y,USD,US,US,IL,SALARIED,TECHNOLOGY_IT,USD_250K_500K,COMPLETED,LOW,2026-05-31,SMS,False
1,CIN0009920001,AFFLUENT,2014-03-09,2014-03-09 11:36:02.071151,None,True,Y,USD,US,US,FL,None,ADMIN_CLERICAL,USD_150K_250K,COMPLETED,LOW,2026-05-08,SMS,False
2,CIN0009920002,MASS,2026-06-06,2026-06-06 16:02:28.967554,None,True,Y,USD,US,US,FL,STUDENT,None,None,COMPLETED,LOW,2026-06-06,SMS,False
3,CIN0009920003,MASS,2019-01-25,2019-01-25 09:09:34.458487,None,True,Y,USD,US,US,MO,SELF_EMPLOYED,FINANCE_INSURANCE,USD_50K_75K,COMPLETED,LOW,2026-03-18,EMAIL,False
4,CIN0009920004,MASS,2022-12-13,2022-12-13 14:03:21.354687,None,True,Y,USD,US,US,OR,RETIRED,None,USD_25K_50K,COMPLETED,LOW,2028-09-03,EMAIL,False


... (14 outputs left out)

📁 session  |  5 file(s)  |  847,108 rows  |  16 columns
session_id: string not null
customer_id: string not null
session_start: timestamp[us] not null
session_end: timestamp[us] not null
session_channel: string not null
device_type: string
os: string
browser: string
app_version: string
login_method: string not null
login_status: string not null
login_failure_reason: string
device_id: string
is_trusted_flag: bool not null
ip_address_hash: string
geo_location_hash: string


,session_id,customer_id,session_start,session_end,session_channel,device_type,os,browser,app_version,login_method,login_status,login_failure_reason,device_id,is_trusted_flag,ip_address_hash,geo_location_hash
0,SES012200000000,CIN0009920000,2025-10-03 19:33:40.032913,2025-10-03 19:44:52.756277,MOBILE_APP,TABLET,IOS,None,7.1.4,BIOMETRIC,SUCCESS,None,302dafbaee0b57b15528ded277ccfac5,False,5db133535eeadae656c31cb295e8ade0,4ef8545fab8a
1,SES012200000001,CIN0009920000,2025-10-24 19:31:18.601133,2025-10-24 19:49:21.080004,MOBILE_APP,TABLET,IOS,None,7.1.4,BIOMETRIC,SUCCESS,None,302dafbaee0b57b15528ded277ccfac5,False,3c69e86d0eaeef0230aed6081bcb2876,4ef8545fab8a
2,SES012200000002,CIN0009920000,2025-10-25 10:39:09.224159,2025-10-25 10:48:48.475074,MOBILE_APP,TABLET,IOS,None,7.1.4,BIOMETRIC,SUCCESS,None,302dafbaee0b57b15528ded277ccfac5,True,3c69e86d0eaeef0230aed6081bcb2876,4ef8545fab8a
3,SES012200000003,CIN0009920000,2025-10-29 19:40:12.632954,2025-10-29 19:45:35.901406,MOBILE_APP,TABLET,IOS,None,7.1.4,PASSWORD,SUCCESS,None,302dafbaee0b57b15528ded277ccfac5,True,3c69e86d0eaeef0230aed6081bcb2876,4ef8545fab8a
4,SES012200000004,CIN0009920000,2025-10-30 16:44:02.806353,2025-10-30 16:47:33.936307,MOBILE_APP,TABLET,IOS,None,7.1.4,BIOMETRIC,SUCCESS,None,302dafbaee0b57b15528ded277ccfac5,True,5db133535eeadae656c31cb295e8ade0,4ef8545fab8a


📁 txn_event  |  5 file(s)  |  10,890,458 rows  |  38 columns
txn_event_id: string not null
record_type: string not null
link_reference: string not null
linked_event_id: string
customer_id: string not null
product_id: string not null
merchant_id: string
event_timestamp: timestamp[us] not null
posted_timestamp: timestamp[us]
settlement_timestamp: timestamp[us]
record_available_timestamp: timestamp[us] not null
amount: decimal128(18, 2) not null
currency: string not null
fx_rate: decimal128(12, 6) not null
balance_before: decimal128(18, 2)
decision: string
posting_status: string
decline_category: string
is_customer_caused: bool
txn_channel: string not null
initiator_type: string
pos_entry_mode: string
transaction_type: string not null
category: string
merchant_country: string
customer_country_code: string not null
is_recurring: bool not null
reversed_transaction_id: string
debit_credit_indicator: string not null
counterparty_type: string not null
account_number_masked: string
payment_fund

,txn_event_id,record_type,link_reference,linked_event_id,customer_id,product_id,merchant_id,event_timestamp,posted_timestamp,settlement_timestamp,record_available_timestamp,amount,currency,fx_rate,balance_before,decision,posting_status,decline_category,is_customer_caused,txn_channel,initiator_type,pos_entry_mode,transaction_type,category,merchant_country,customer_country_code,is_recurring,reversed_transaction_id,debit_credit_indicator,counterparty_type,account_number_masked,payment_funding_product,cvv_result,pin_verified_flag,threeds_flag,threeds_outcome,is_fraud_flagged,alert_id
0,TXN01200100000163,AUTHORIZATION,0120000000001,None,CIN0009920000,PRD0160000003,MER10010032,2025-10-02 20:33:29.443122,NaT,NaT,2025-10-02 20:33:30.413495,12.08,USD,1.000000,None,DECLINED,None,EXPIRED_CARD,True,POS,None,CHIP,PURCHASE,DINING,US,US,False,None,D,MERCHANT,****0217,None,None,True,None,None,False,None
1,TXN01200100000164,AUTHORIZATION,0120000000002,None,CIN0009920000,PRD0160000003,MER10001835,2025-10-03 12:43:37.675661,NaT,NaT,2025-10-03 12:43:37.850591,23.95,USD,1.000000,None,APPROVED,None,None,None,POS,None,CHIP,PURCHASE,SHOPPING,US,US,False,None,D,MERCHANT,****0217,None,None,True,None,None,False,None
2,TXN01200100000165,POSTING,0120000000002,TXN01200100000164,CIN0009920000,PRD0160000003,MER10001835,2025-10-06 14:09:37.359484,2025-10-06 14:09:37.359484,2025-10-08 16:30:10.864541,2025-10-07 08:34:36.887089,23.95,USD,1.000000,309.40,None,POSTED,None,None,POS,CUSTOMER,None,PURCHASE,SHOPPING,US,US,False,None,D,MERCHANT,****0217,None,None,None,None,None,False,None
3,TXN01200100000166,POSTING,0120000000003,None,CIN0009920000,PRD0160000003,MER10013249,2025-10-05 19:08:13.831607,2025-10-05 19:08:13.831607,2025-10-07 00:20:12.370164,2025-10-06 06:02:07.901692,22.10,USD,1.000000,291.50,None,POSTED,None,None,ECOMMERCE,CUSTOMER,None,PURCHASE,BILLS_UTILITIES,US,US,True,None,D,MERCHANT,****0217,None,None,None,None,None,False,None
4,TXN01200100000167,POSTING,FNDW012000000000167,None,CIN0009920000,PRD0160000003,None,2025-10-03 19:44:49.256961,2025-10-03 19:44:49.256961,2025-10-03 19:44:49.256961,2025-10-03 21:44:49.256961,100.00,USD,1.000000,51.50,None,POSTED,None,None,MOBILE_APP,CUSTOMER,None,TRANSFER_IN,TRANSFERS,None,US,False,None,C,OWN_ACCOUNT,****0217,None,None,None,None,None,False,None


## 3. Load one full table
Change `TABLE` to any subfolder name. Watch out: `txn_event` (~380 MB) and `interaction_event` (~160 MB) are big.

In [11]:
TABLE = "customer_profile"

df = pq.read_table(f"{ROOT}/{TABLE}", filesystem=s3).to_pandas()
print(df.shape)
df.head()

(10000, 19)


,customer_id,cust_segment,relationship_open_date,acc_cre_ts,relationship_close_date,customer_is_active,is_cust_stat_clr_y_n,home_currency,nationality_country_code,residence_country,residence_state,employment_status,occupation_category,income_band,kyc_status,kyc_risk_rating,kyc_last_review_date,preferred_contact_channel,marketing_consent
0,CIN0009920000,AFFLUENT,2016-02-28,2016-02-28 12:23:00.690962,None,True,Y,USD,US,US,IL,SALARIED,TECHNOLOGY_IT,USD_250K_500K,COMPLETED,LOW,2026-05-31,SMS,False
1,CIN0009920001,AFFLUENT,2014-03-09,2014-03-09 11:36:02.071151,None,True,Y,USD,US,US,FL,None,ADMIN_CLERICAL,USD_150K_250K,COMPLETED,LOW,2026-05-08,SMS,False
2,CIN0009920002,MASS,2026-06-06,2026-06-06 16:02:28.967554,None,True,Y,USD,US,US,FL,STUDENT,None,None,COMPLETED,LOW,2026-06-06,SMS,False
3,CIN0009920003,MASS,2019-01-25,2019-01-25 09:09:34.458487,None,True,Y,USD,US,US,MO,SELF_EMPLOYED,FINANCE_INSURANCE,USD_50K_75K,COMPLETED,LOW,2026-03-18,EMAIL,False
4,CIN0009920004,MASS,2022-12-13,2022-12-13 14:03:21.354687,None,True,Y,USD,US,US,OR,RETIRED,None,USD_25K_50K,COMPLETED,LOW,2028-09-03,EMAIL,False


In [12]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max
customer_id,10000,10000,CIN0009929983,1,NaN,NaN,NaN,NaN,NaN,NaN
cust_segment,10000,4,MASS,7198,NaN,NaN,NaN,NaN,NaN,NaN
relationship_open_date,10000,5316,2012-10-10,16,NaN,NaN,NaN,NaN,NaN,NaN
acc_cre_ts,10000,NaN,NaN,NaN,2019-04-10 12:29:24.586029,2003-11-20 10:20:24.034678,2015-02-05 12:18:39.891770,2019-12-03 13:10:31.109985,2024-01-10 03:19:12.932789,2028-09-10 12:03:41.314878
relationship_close_date,784,699,2028-07-30,4,NaN,NaN,NaN,NaN,NaN,NaN
customer_is_active,10000,2,True,9216,NaN,NaN,NaN,NaN,NaN,NaN
is_cust_stat_clr_y_n,10000,2,Y,9786,NaN,NaN,NaN,NaN,NaN,NaN
home_currency,10000,6,USD,9943,NaN,NaN,NaN,NaN,NaN,NaN
nationality_country_code,9717,35,US,8434,NaN,NaN,NaN,NaN,NaN,NaN
residence_country,10000,6,US,9924,NaN,NaN,NaN,NaN,NaN,NaN
